# Training notebook

This is a cleaned-up notebook used for training and testing individual RF, MLP and CNN models. This notebook has now been updated to train on R1a, R1b and R2 regions, holding out the R3 region as a validation check. This is a better metric to use for validation than the random sample that was done before, as it avoids the problem of pixels in the training set being correlated with those in the validation set.

This notebook does not load PCs, since it has now been established that this gives a reduction in performance.



In [1]:
import os
import gc
import time
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree

# ── Constants ────────────────────────────────────────────────
RANDOM_STATE  = 42
PARQUET_DIR   = "/nvme1/users/md962/glacier/Glacier Project/Merged/"
MODELS_DIR    = "/nvme1/users/md962/glacier/Glacier Project/Final_Models/"
RESULTS_DIR   = "/nvme1/users/md962/glacier/Glacier Project/Results/"
TENSORS_DIR   = "/nvme1/users/md962/glacier/Glacier Project/Tensors/"
PREDS_DIR     = "/nvme1/users/md962/glacier/Glacier Project/Predictions/"
TRAIN_REGIONS = ['r1a', 'r1b', 'r2']
VAL_REGION    = 'r3'
AE_COLS       = [f'A{i:02d}' for i in range(64)]
EASD_COLS     = ['elevation', 'edge_distance', 'aspect', 'slope']
RF_SAMPLE_N   = 2_000_000

for d in [MODELS_DIR, RESULTS_DIR, TENSORS_DIR, PREDS_DIR]:
    os.makedirs(d, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Workstation Edition


In [3]:
# Load r3 first so its pixels take priority in dedup
dfs = []
for region in ['r3', 'r1a', 'r1b', 'r2']:  # r3 first
    df = pd.read_parquet(f"{PARQUET_DIR}{region}_combined.parquet")
    df['region'] = region
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
df_all = df_all.drop_duplicates(subset=['lon', 'lat'], keep='first').reset_index(drop=True)

print(f"Total unique pixels: {len(df_all):,}")
for region in ['r1a', 'r1b', 'r2', 'r3']:
    df_r = df_all[df_all['region'] == region]
    print(f"  {region}: {len(df_r):,} pixels, melt rate: {df_r['melt_label'].mean():.3f}")

Total unique pixels: 13,996,828
  r1a: 6,222,913 pixels, melt rate: 0.139
  r1b: 1,028,089 pixels, melt rate: 0.264
  r2: 4,971,507 pixels, melt rate: 0.239
  r3: 1,774,319 pixels, melt rate: 0.240


In [4]:
df_train = df_all[df_all['region'].isin(['r1a', 'r1b', 'r2'])].reset_index(drop=True)
df_val   = df_all[df_all['region'] == 'r3'].reset_index(drop=True)

print(f"Train pixels: {len(df_train):,}, melt rate: {df_train['melt_label'].mean():.3f}")
print(f"Val pixels:   {len(df_val):,},   melt rate: {df_val['melt_label'].mean():.3f}")

Train pixels: 12,222,509, melt rate: 0.190
Val pixels:   1,774,319,   melt rate: 0.240


In [5]:
print(f"Sampling {RF_SAMPLE_N:,} pixels for RF (stratified)...")
df_rf_sample, _ = train_test_split(
    df_train,
    train_size=RF_SAMPLE_N,
    stratify=df_train['melt_label'],
    random_state=RANDOM_STATE
)
print(f"Sample melt rate: {df_rf_sample['melt_label'].mean():.3f}")

X_train_easd = df_rf_sample[EASD_COLS].values.astype(np.float32)
y_train_rf   = df_rf_sample['melt_label'].values
X_val_easd   = df_val[EASD_COLS].values.astype(np.float32)
y_val        = df_val['melt_label'].values

def cm_errors(cm):
    ice  = 1 - cm[0, 0] / cm[0].sum()
    melt = 1 - cm[1, 1] / cm[1].sum()
    return ice, melt

print("\nTraining RF...")
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=300, oob_score=True,
    n_jobs=-1, random_state=RANDOM_STATE
)
rf.fit(X_train_easd, y_train_rf)
print(f"Trained in {time.time()-t0:.1f}s")

oob_ice, oob_melt = cm_errors(
    confusion_matrix(y_train_rf, rf.oob_decision_function_.argmax(axis=1))
)
print(f"OOB:  Ice={oob_ice:.4f}, Melt={oob_melt:.4f}")

val_probs_rf = rf.predict_proba(X_val_easd)[:, 1]
val_preds_rf = rf.predict(X_val_easd)
val_ice, val_melt = cm_errors(confusion_matrix(y_val, val_preds_rf))
val_auc_rf = roc_auc_score(y_val, val_probs_rf)
print(f"Val:  Ice={val_ice:.4f}, Melt={val_melt:.4f}, AUC={val_auc_rf:.4f}")

joblib.dump(rf, f"{MODELS_DIR}RF_EASD_R3HOLDOUT.joblib")
print("RF saved")
del X_train_easd, y_train_rf, df_rf_sample
gc.collect()

Sampling 2,000,000 pixels for RF (stratified)...
Sample melt rate: 0.190

Training RF...
Trained in 57.3s
OOB:  Ice=0.0441, Melt=0.3005
Val:  Ice=0.0376, Melt=0.4853, AUC=0.8987
RF saved


58

In [13]:
ROUND_DP = 6

def build_neighbour_table(df, patch_size=3, tolerance=0.6):
    half   = patch_size // 2
    coords = np.stack([df['lon'].values, df['lat'].values], axis=1)
    pixel_lon = np.median(np.diff(np.sort(df['lon'].unique())))
    pixel_lat = np.median(np.diff(np.sort(df['lat'].unique())))
    print(f"  Building KD-tree for {len(df):,} pixels...")
    tree = cKDTree(coords)
    k = patch_size ** 2 + 1
    distances, indices = tree.query(coords, k=k, workers=-1)
    neighbour_table = np.full((len(df), patch_size, patch_size), -1, dtype=np.int32)
    
    for slot in range(k):
        neighbour_coords = coords[indices[:, slot]]
        delta_lon = neighbour_coords[:, 0] - coords[:, 0]
        delta_lat = neighbour_coords[:, 1] - coords[:, 1]
        dc = np.round(delta_lon / pixel_lon).astype(int)
        dr = np.round(delta_lat / pixel_lat).astype(int)
        
    # Always compute patch_row/col before validity check
        patch_row = (dr + half).clip(0, patch_size - 1)
        patch_col = (dc + half).clip(0, patch_size - 1)
        
        residual_lon = np.abs(delta_lon - dc * pixel_lon)
        residual_lat = np.abs(delta_lat - dr * pixel_lat)
        
        valid = (
            (np.abs(dc) <= half) & (np.abs(dr) <= half) &
            (residual_lon < tolerance * pixel_lon) &
            (residual_lat < tolerance * pixel_lat) &
            (patch_row >= 0) & (patch_row < patch_size) &
            (patch_col >= 0) & (patch_col < patch_size)
        )
        
        pixel_indices = np.arange(len(df))
        mask = valid & (neighbour_table[pixel_indices, patch_row, patch_col] == -1)
        neighbour_table[pixel_indices[mask], patch_row[mask], patch_col[mask]] = \
            indices[mask, slot]
    
    return neighbour_table, pixel_lon, pixel_lat


def build_gpu_tensors_pixel(df):
    """Single pixel — no neighbourhood. Input shape: (N, 64)"""
    embeddings = df[AE_COLS].values.astype(np.float32)   # (N, 64)
    labels     = df['melt_label'].values.astype(np.float32)
    print(f"  Moving {len(df):,} pixels to GPU...")
    return TensorDataset(
        torch.from_numpy(embeddings).cuda(),
        torch.from_numpy(labels).cuda()
    )

def build_gpu_tensors_patch(df, patch_size=3):
    """3x3 patch — for CNN. Input shape: (N, 64, 3, 3)"""
    nt, _, _ = build_neighbour_table(df, patch_size)
    n    = len(df)
    flat = nt.reshape(n, patch_size * patch_size)
    missing   = flat == -1
    flat_safe = flat.copy()
    flat_safe[missing] = 0
    embeddings = df[AE_COLS].values.astype(np.float32)
    patches    = embeddings[flat_safe]        # (N, 9, 64)
    patches[missing] = 0.0
    patches = patches.transpose(0, 2, 1).reshape(n, 64, patch_size, patch_size)
    labels  = df['melt_label'].values.astype(np.float32)
    print(f"  Moving {n:,} patches to GPU...")
    return TensorDataset(
        torch.from_numpy(patches).cuda(),
        torch.from_numpy(labels).cuda()
    )

class GPUDataLoader:
    def __init__(self, tensor_dataset, batch_size, shuffle=True):
        self.patches    = tensor_dataset.tensors[0]
        self.labels     = tensor_dataset.tensors[1]
        self.batch_size = batch_size
        self.shuffle    = shuffle
        self.n          = len(self.labels)
        self.dataset    = self
    def __len__(self):
        return (self.n + self.batch_size - 1) // self.batch_size
    def __iter__(self):
        idx = torch.randperm(self.n, device='cuda') if self.shuffle \
              else torch.arange(self.n, device='cuda')
        for start in range(0, self.n, self.batch_size):
            batch_idx = idx[start:start + self.batch_size]
            yield self.patches[batch_idx], self.labels[batch_idx]

def run_epoch(model, loader, criterion, optimizer=None, train=True):
    model.train() if train else model.eval()
    total_loss = 0
    all_labels, all_probs = [], []
    with torch.set_grad_enabled(train):
        for patches, labels in loader:
            logits = model(patches)
            loss   = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            all_probs.extend(torch.sigmoid(logits).cpu().detach().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / loader.n
    auc      = roc_auc_score(all_labels, all_probs)
    return avg_loss, auc

def train_model(model, train_loader, val_loader, pos_weight,
                n_epochs=20, patience=5, save_path=None):
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2
    )
    best_val_auc = 0
    best_epoch   = 0
    print(f"\n{'Epoch':>5}  {'Train Loss':>10}  {'Train AUC':>9}  "
          f"{'Val Loss':>8}  {'Val AUC':>7}")
    print("-" * 55)
    for epoch in range(1, n_epochs + 1):
        train_loss, train_auc = run_epoch(model, train_loader, criterion, optimizer, train=True)
        val_loss,   val_auc   = run_epoch(model, val_loader,   criterion, train=False)
        scheduler.step(val_auc)
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch   = epoch
            if save_path:
                torch.save(model.state_dict(), save_path)
        print(f"{epoch:>5}  {train_loss:>10.4f}  {train_auc:>9.4f}  "
              f"{val_loss:>8.4f}  {val_auc:>7.4f}"
              + (" ← best" if epoch == best_epoch else ""))
        if epoch - best_epoch >= patience:
            print(f"\nEarly stopping at epoch {epoch}")
            break
    print(f"\nBest val AUC: {best_val_auc:.4f} at epoch {best_epoch}")
    return best_val_auc

def predict_pytorch(model, tensor_dataset, batch_size=65536):
    """Run inference on a TensorDataset, return probability array."""
    model.eval()
    loader = GPUDataLoader(tensor_dataset, batch_size=batch_size, shuffle=False)
    probs  = []
    with torch.no_grad():
        for patches, _ in loader:
            logits = model(patches)
            probs.extend(torch.sigmoid(logits).cpu().numpy())
    return np.array(probs, dtype=np.float32)


In [7]:
# Single pixel tensors for MLP
print("Building single-pixel tensors (MLP)...")
mlp_train_tensors = {}
for region in TRAIN_REGIONS:
    print(f"  {region}...")
    df = pd.read_parquet(f"{PARQUET_DIR}{region}_combined.parquet")
    df = df.drop_duplicates(subset=['lon', 'lat'], keep='first')
    mlp_train_tensors[region] = build_gpu_tensors_pixel(df)
    torch.save(mlp_train_tensors[region], f"{TENSORS_DIR}mlp_{region}.pt")
    del df; gc.collect()

mlp_val_tensors = build_gpu_tensors_pixel(df_val)
torch.save(mlp_val_tensors, f"{TENSORS_DIR}mlp_r3.pt")

# Patch tensors for CNN
print("\nBuilding patch tensors (CNN)...")
cnn_train_tensors = {}
for region in TRAIN_REGIONS:
    print(f"  {region}...")
    df = pd.read_parquet(f"{PARQUET_DIR}{region}_combined.parquet")
    df = df.drop_duplicates(subset=['lon', 'lat'], keep='first')
    cnn_train_tensors[region] = build_gpu_tensors_patch(df, patch_size=3)
    torch.save(cnn_train_tensors[region], f"{TENSORS_DIR}cnn_{region}.pt")
    del df; gc.collect()

cnn_val_tensors = build_gpu_tensors_patch(df_val, patch_size=3)
torch.save(cnn_val_tensors, f"{TENSORS_DIR}cnn_r3.pt")
print("\nAll tensors built and saved")

Building single-pixel tensors (MLP)...
  r1a...
  Moving 6,222,913 pixels to GPU...
  r1b...
  Moving 1,028,089 pixels to GPU...
  r2...
  Moving 6,217,318 pixels to GPU...
  Moving 1,774,319 pixels to GPU...

Building patch tensors (CNN)...
  r1a...
  Building KD-tree for 6,222,913 pixels...
  Moving 6,222,913 patches to GPU...
  r1b...
  Building KD-tree for 1,028,089 pixels...
  Moving 1,028,089 patches to GPU...
  r2...
  Building KD-tree for 6,217,318 pixels...
  Moving 6,217,318 patches to GPU...
  Building KD-tree for 1,774,319 pixels...
  Moving 1,774,319 patches to GPU...

All tensors built and saved


In [8]:
def make_loaders(train_tensors_dict, val_tensors, batch_size=65536):
    # Concatenate training regions
    all_patches = torch.cat([train_tensors_dict[r].tensors[0] for r in TRAIN_REGIONS])
    all_labels  = torch.cat([train_tensors_dict[r].tensors[1] for r in TRAIN_REGIONS])
    train_combined = TensorDataset(all_patches, all_labels)
    
    # pos_weight from training labels
    melt_rate  = all_labels.cpu().mean().item()
    pos_weight = torch.tensor((1 - melt_rate) / melt_rate)
    print(f"  Melt rate: {melt_rate:.3f}, pos_weight: {pos_weight:.2f}")
    print(f"  GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
    
    return (
        GPUDataLoader(train_combined, batch_size=batch_size, shuffle=True),
        GPUDataLoader(val_tensors,    batch_size=batch_size, shuffle=False),
        pos_weight
    )

print("MLP loaders:")
mlp_train_loader, mlp_val_loader, mlp_pos_weight = make_loaders(
    mlp_train_tensors, mlp_val_tensors
)

print("\nCNN loaders:")
cnn_train_loader, cnn_val_loader, cnn_pos_weight = make_loaders(
    cnn_train_tensors, cnn_val_tensors
)

MLP loaders:
  Melt rate: 0.189, pos_weight: 4.29
  GPU memory: 42.6 GB

CNN loaders:
  Melt rate: 0.189, pos_weight: 4.29
  GPU memory: 73.7 GB


In [9]:
class GlacierMLP(nn.Module):
    """
    Single-pixel MLP baseline.
    Input:  (B, 64) — raw AlphaEarth embedding
    Output: (B,)    — raw logit
    """
    def __init__(self, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

mlp_model = GlacierMLP(dropout=0.5).to(device)
dummy = torch.zeros(8, 64).to(device)
print(f"MLP output shape: {mlp_model(dummy).shape}")
print(f"MLP parameters: {sum(p.numel() for p in mlp_model.parameters()):,}")

best_mlp_auc = train_model(
    mlp_model,
    mlp_train_loader,
    mlp_val_loader,
    mlp_pos_weight,
    n_epochs=20,
    patience=5,
    save_path=f"{MODELS_DIR}MLP_AE64_R3HOLDOUT.pt"
)

MLP output shape: torch.Size([8])
MLP parameters: 199,425

Epoch  Train Loss  Train AUC  Val Loss  Val AUC
-------------------------------------------------------
    1      0.4843     0.9447    0.7849   0.9239 ← best
    2      0.4284     0.9555    0.7745   0.9277 ← best
    3      0.4111     0.9588    0.7585   0.9329 ← best
    4      0.3985     0.9611    0.7745   0.9370 ← best
    5      0.3892     0.9628    0.7628   0.9400 ← best
    6      0.3824     0.9640    0.7419   0.9405 ← best
    7      0.3774     0.9648    0.7984   0.9415 ← best
    8      0.3733     0.9655    0.7748   0.9421 ← best
    9      0.3702     0.9661    0.7531   0.9405
   10      0.3677     0.9665    0.7847   0.9421 ← best
   11      0.3655     0.9668    0.7772   0.9433 ← best
   12      0.3635     0.9672    0.8247   0.9420
   13      0.3622     0.9674    0.8198   0.9427
   14      0.3609     0.9676    0.7934   0.9432
   15      0.3553     0.9685    0.7854   0.9444 ← best
   16      0.3535     0.9688    0.8084  

In [10]:
class GlacierCNN(nn.Module):
    """
    Patch-based CNN.
    Input:  (B, 64, 3, 3) — 3x3 AlphaEarth patch
    Output: (B,)           — raw logit
    """
    def __init__(self, dropout=0.5):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=2),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=2),
            nn.BatchNorm2d(256),
            nn.ReLU(),
        )
        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.fc_block(self.conv_block(x)).squeeze(1)

cnn_model = GlacierCNN(dropout=0.5).to(device)
dummy = torch.zeros(8, 64, 3, 3).to(device)
print(f"CNN output shape: {cnn_model(dummy).shape}")
print(f"CNN parameters: {sum(p.numel() for p in cnn_model.parameters()):,}")

best_cnn_auc = train_model(
    cnn_model,
    cnn_train_loader,
    cnn_val_loader,
    cnn_pos_weight,
    n_epochs=20,
    patience=5,
    save_path=f"{MODELS_DIR}CNN_PATCH3_R3HOLDOUT.pt"
)

CNN output shape: torch.Size([8])
CNN parameters: 202,049

Epoch  Train Loss  Train AUC  Val Loss  Val AUC
-------------------------------------------------------
    1      0.4417     0.9548    0.7774   0.9302 ← best
    2      0.3585     0.9687    0.8326   0.9445 ← best
    3      0.3337     0.9725    1.0084   0.9452 ← best
    4      0.3188     0.9747    0.8655   0.9435
    5      0.3060     0.9765    1.1586   0.9454 ← best
    6      0.2977     0.9776    1.3713   0.9473 ← best
    7      0.2900     0.9787    1.1377   0.9430
    8      0.2851     0.9793    1.0935   0.9465
    9      0.2792     0.9801    1.3578   0.9482 ← best
   10      0.2757     0.9806    1.2554   0.9439
   11      0.2708     0.9812    1.2131   0.9427
   12      0.2681     0.9815    1.2206   0.9475
   13      0.2559     0.9830    1.3061   0.9441
   14      0.2540     0.9832    1.1081   0.9451

Early stopping at epoch 14

Best val AUC: 0.9482 at epoch 9


In [14]:
# Use already-deduplicated df_all instead of reloading
df_meta = df_all[['lon', 'lat', 'melt_label', 'edge_distance', 'region']].copy()

all_probs_rf  = []
all_probs_mlp = []
all_probs_cnn = []

for region in ['r3', 'r1a', 'r1b', 'r2']:
    print(f"\n{region}...")
    df = df_all[df_all['region'] == region].reset_index(drop=True)
    
    # RF predictions
    rf_p = rf.predict_proba(df[EASD_COLS].values.astype(np.float32))[:, 1]
    
    # MLP predictions
    mlp_tensors = build_gpu_tensors_pixel(df)
    mlp_p = predict_pytorch(mlp_model, mlp_tensors)
    del mlp_tensors; torch.cuda.empty_cache()
    
    # CNN predictions
    cnn_tensors = build_gpu_tensors_patch(df, patch_size=3)
    cnn_p = predict_pytorch(cnn_model, cnn_tensors)
    del cnn_tensors; torch.cuda.empty_cache()
    
    all_probs_rf.append(rf_p.astype(np.float32))
    all_probs_mlp.append(mlp_p)
    all_probs_cnn.append(cnn_p)
    
    print(f"  RF mean: {rf_p.mean():.3f}, MLP mean: {mlp_p.mean():.3f}, "
          f"CNN mean: {cnn_p.mean():.3f}")

# Combine — reindex df_meta to match region order
df_meta = pd.concat([
    df_all[df_all['region'] == r] for r in ['r3', 'r1a', 'r1b', 'r2']
], ignore_index=True)[['lon', 'lat', 'melt_label', 'edge_distance', 'region']]

df_preds = df_meta.copy()
df_preds['prob_rf_easd']    = np.concatenate(all_probs_rf)
df_preds['prob_mlp_ae64']   = np.concatenate(all_probs_mlp)
df_preds['prob_cnn_patch3'] = np.concatenate(all_probs_cnn)

print(f"\nTotal predictions: {len(df_preds):,}")
df_preds.to_parquet(
    f"{PREDS_DIR}predictions_full_peru_r3holdout.parquet",
    index=False
)
print("Saved")


r3...
  Moving 1,774,319 pixels to GPU...
  Building KD-tree for 1,774,319 pixels...
  Moving 1,774,319 patches to GPU...
  RF mean: 0.174, MLP mean: 0.222, CNN mean: 0.242

r1a...
  Moving 6,222,913 pixels to GPU...
  Building KD-tree for 6,222,913 pixels...
  Moving 6,222,913 patches to GPU...
  RF mean: 0.147, MLP mean: 0.199, CNN mean: 0.195

r1b...
  Moving 1,028,089 pixels to GPU...
  Building KD-tree for 1,028,089 pixels...
  Moving 1,028,089 patches to GPU...
  RF mean: 0.273, MLP mean: 0.351, CNN mean: 0.344

r2...
  Moving 4,971,507 pixels to GPU...
  Building KD-tree for 4,971,507 pixels...
  Moving 4,971,507 patches to GPU...
  RF mean: 0.231, MLP mean: 0.311, CNN mean: 0.305

Total predictions: 13,996,828
Saved


In [16]:
MODELS = {
    'RF (EASD)':    'prob_rf_easd',
    'MLP (AE64)':   'prob_mlp_ae64',
    'CNN (patch3)': 'prob_cnn_patch3'
}

def overlap_table(df, title):
    actual = df['melt_label'].values == 1
    n_melt = actual.sum()
    print(f"\n=== {title} ===")
    print(f"Pixels: {len(df):,}, Melt: {n_melt:,} ({n_melt/len(df)*100:.1f}%)")
    print(f"{'Model':<20} {'Overlap %':>10} {'IoU':>8} {'AUC':>8}")
    print("-" * 50)
    for name, col in MODELS.items():
        probs = df[col].values
        top   = np.argsort(probs)[::-1][:n_melt]
        pred  = np.zeros(len(probs), dtype=bool)
        pred[top] = True
        n_int = (pred & actual).sum()
        ovlp  = n_int / n_melt * 100
        iou   = n_int / (pred | actual).sum()
        auc   = roc_auc_score(actual, probs)
        print(f"{name:<20} {ovlp:>9.2f}% {iou:>8.4f} {auc:>8.4f}")

overlap_table(df_preds, "PERU-WIDE (includes training regions)")
overlap_table(df_preds[df_preds['region'] == 'r3'], "r3 ONLY (out-of-sample for all models)")


=== PERU-WIDE (includes training regions) ===
Pixels: 13,996,828, Melt: 2,751,035 (19.7%)
Model                 Overlap %      IoU      AUC
--------------------------------------------------
RF (EASD)                77.82%   0.6369   0.9528
MLP (AE64)               81.10%   0.6821   0.9698
CNN (patch3)             84.54%   0.7322   0.9773

=== r3 ONLY (out-of-sample for all models) ===
Pixels: 1,774,319, Melt: 426,115 (24.0%)
Model                 Overlap %      IoU      AUC
--------------------------------------------------
RF (EASD)                70.29%   0.5418   0.8987
MLP (AE64)               79.52%   0.6600   0.9454
CNN (patch3)             79.85%   0.6646   0.9451
